# 🐄 ĐỀ TÀI DINH DƯỠNG – SO SÁNH MÔ HÌNH CHƯA CHỈNH SỬA vs ĐÃ CẢI TIẾN

Notebook này chạy trên **Google Colab (CPU)**, tải bộ dữ liệu công khai gốc từ GitHub,
huấn luyện và **so sánh trực tiếp**:

| | Cấu hình | Kết quả dự kiến (luận văn) |
|---|---|---|
| ❌ Chưa chỉnh sửa | **RF 500 cây** – phương pháp của bài báo gốc Pongsanun et al. 2025 | acc 99,89%; macro-F1 0,9973 |
| ❌ Chưa chỉnh sửa | RF + chọn 60 đặc trưng kiểu RFE | acc 99,90%; macro-F1 0,9975 |
| ✅ Đã cải tiến | **HGB gộp đàn** (HistGradientBoosting) | acc 99,95%; macro-F1 0,9982 |
| ✅ Đã cải tiến | **HGB + ensemble 6 chuyên gia OvR** (được chọn) | macro-F1 **0,9983**; F1 đi lại **0,9933** |
| ⚠️ Ví dụ âm | RankMix trộn hạng giữa các lớp | acc sụp còn 81,5% |

Dữ liệu: 7.295 cửa sổ × 142 cột, 6 lớp hành vi; **lớp đi lại chỉ 1,8% (133 mẫu)** —
đây chính là điểm yếu của bài báo gốc và là mục tiêu của bản sửa.
Thời gian chạy toàn bộ khoảng **5–8 phút** trên Colab CPU.

**Cách dùng:** upload file này lên Colab → Runtime → Run all.

In [ ]:
import time, requests
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier as HGB
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
RNG = 42; np.random.seed(RNG)
print("Sẵn sàng.")

In [ ]:
# ---------- TẢI DỮ LIỆU CÔNG KHAI GỐC TỪ GITHUB (kèm bài báo PMC12632110) ----------
URL = "https://raw.githubusercontent.com/pongsanun/cow-behavior-classification-acc-gyro-combined/main/cow_data_processed_updated.csv"
r = requests.get(URL, timeout=180); r.raise_for_status()
open("cow_data_processed_updated.csv", "wb").write(r.content)
print("Đã tải", len(r.content), "bytes")

In [ ]:
# ---------- TIỀN XỬ LÝ GIỐNG HỆT THỰC NGHIỆM CỦA LUẬN VĂN ----------
df = pd.read_csv("cow_data_processed_updated.csv")
df["behav"] = df["Classification"].replace({"walking ruminating": "walking"})
classes = ["lying", "lying ruminating", "standing", "standing ruminating", "eating", "walking"]
df = df[df["behav"].isin(classes)].copy()
le = LabelEncoder(); df["y"] = le.fit_transform(df["behav"])
EAT  = int(np.where(le.classes_ == "eating")[0][0])   # ma lop AN (chu y: LabelEncoder xep alphabet, eating = 0)
WALK = int(np.where(le.classes_ == "walking")[0][0])  # ma lop DI LAI

# LOẠI các cột rò rỉ: các cột nhị phân này là bản tóm tắt của chính nhãn
leak = ["Lying", "Lying down", "Standing", "Standing up", "Walking", "Feeding",
        "Dringking", "Ruminating", "Other", "y"]
raw = ["accX","accY","accZ","gyroX","gyroY","gyroZ","Sensor","Time","ID"] + classes + ["Classification","behav"] + leak
feat = [x for x in df.columns if x not in raw and df[x].dtype != object]
X = df[feat].astype(float); y = df["y"].values
print("Mẫu:", len(y), "| Đặc trưng:", len(feat))
print("\nPhân bố lớp (chú ý lớp đi lại chỉ 1,8%):")
print(pd.Series(le.classes_[y]).value_counts())

---
## GIAO THỨC & CHỈ SỐ
- Kiểm định chéo **10 lượt phân tầng** (StratifiedKFold); mỗi mẫu được chấm đúng một lần.
- Chỉ số chính: **macro-F1** — trung bình F1 của từng lớp không trọng số,
  phạt nặng mô hình bỏ rơi lớp hiếm (đi lại 1,8%).

In [ ]:
skf = StratifiedKFold(10, shuffle=True, random_state=RNG)
folds = list(skf.split(X, y))

def pipe(m, scaled=False):
    st = [("imp", SimpleImputer(strategy="median"))]
    if scaled: st.append(("sc", StandardScaler()))
    st.append(("clf", m)); return Pipeline(st)

def oof_eval(tag, fn, scaled=False):
    P = np.zeros((len(y), 6)); t0 = time.time()
    for tr, te in folds:
        m = pipe(fn(), scaled); m.fit(X.iloc[tr], y[tr])
        P[te] = m.predict_proba(X.iloc[te])
    pred = P.argmax(1)
    r = {"tag": tag, "acc": accuracy_score(y, pred), "macroF1": f1_score(y, pred, average="macro"),
         "F1_eating": f1_score(y == EAT, pred == EAT), "F1_walking": f1_score(y == WALK, pred == WALK)}
    print("%-42s acc=%.4f  macroF1=%.4f  F1_an=%.4f  F1_di=%.4f  (%.0fs)"
          % (tag, r["acc"], r["macroF1"], r["F1_eating"], r["F1_walking"], time.time()-t0))
    return P, r

res = {}

In [ ]:
# ================= MÔ HÌNH CHƯA CHỈNH SỬA – PHƯƠNG PHÁP BÀI BÁO GỐC =================
# (1) Random Forest 500 cây (bài báo dùng RF per-cow; dữ liệu công khai không có mã bò
#     nên tái hiện trên toàn đàn – cách so sánh công bằng nhất có thể)
P_rf, res["RF_baiBao"] = oof_eval("RF 500 cây – phương pháp bài báo",
                                  lambda: RandomForestClassifier(500, random_state=RNG, n_jobs=-1))

# (2) RF + chọn 60 đặc trưng quan trọng nhất kiểu RFE
rf0 = RandomForestClassifier(500, random_state=RNG, n_jobs=-1).fit(X, y)
idx60 = np.argsort(rf0.feature_importances_)[::-1][:60]
P60 = np.zeros((len(y), 6))
for tr, te in folds:
    m = RandomForestClassifier(500, random_state=RNG, n_jobs=-1)
    m.fit(X.iloc[tr, idx60], y[tr]); P60[te] = m.predict_proba(X.iloc[te, idx60])
pred = P60.argmax(1)
res["RF_RFE60"] = {"tag": "RF + RFE 60 đặc trưng", "acc": accuracy_score(y, pred),
                   "macroF1": f1_score(y, pred, average="macro"),
                   "F1_eating": f1_score(y == EAT, pred == EAT), "F1_walking": f1_score(y == WALK, pred == WALK)}
d = res["RF_RFE60"]
print("%-42s acc=%.4f  macroF1=%.4f  F1_an=%.4f  F1_di=%.4f" % (d["tag"], d["acc"], d["macroF1"], d["F1_eating"], d["F1_walking"]))

In [ ]:
# ================= MÔ HÌNH ĐÃ CẢI TIẾN =================
# (3) Sửa 1+2: gộp đàn + HistGradientBoosting (boosting dồn sức vào mẫu sai -> thân thiện lớp hiếm)
P_hgb, res["HGB"] = oof_eval("HGB gộp đàn (ĐÃ SỬA)", lambda: HGB(random_state=RNG, max_iter=300))

# (4) Sửa 3: ensemble 6 chuyên gia OvR – mỗi hành vi một mô hình nhị phân riêng
P_ovr = np.zeros((len(y), 6)); t0 = time.time()
for cl in range(6):
    yc = (y == cl).astype(int)
    for tr, te in folds:
        m = HGB(random_state=RNG, max_iter=300)
        m.fit(X.iloc[tr], yc[tr]); P_ovr[te, cl] = m.predict_proba(X.iloc[te])[:, 1]
pred = P_ovr.argmax(1)
res["OvR"] = {"tag": "HGB + 6 chuyên gia OvR (ĐÃ SỬA – được chọn)", "acc": accuracy_score(y, pred),
              "macroF1": f1_score(y, pred, average="macro"),
              "F1_eating": f1_score(y == EAT, pred == EAT), "F1_walking": f1_score(y == WALK, pred == WALK)}
d = res["OvR"]
print("%-42s acc=%.4f  macroF1=%.4f  F1_an=%.4f  F1_di=%.4f  (%.0fs)"
      % (d["tag"], d["acc"], d["macroF1"], d["F1_eating"], d["F1_walking"], time.time()-t0))

In [ ]:
# ================= VÍ DỤ ÂM: RANKMIX (thử và LOẠI) =================
# Ý tưởng: trộn HẠNG giữa 6 lớp của HGB và OvR rồi argmax.
# Thất bại vì hạng của từng lớp có phân bố khác nhau – không so sánh được.
def rk(a): return rankdata(a) / (len(a) + 1)
P_mix = 0.5*np.array([rk(P_hgb[:, k]) for k in range(6)]).T + 0.5*np.array([rk(P_ovr[:, k]) for k in range(6)]).T
pred = P_mix.argmax(1)
res["RankMix"] = {"tag": "RankMix (BỊ LOẠI)", "acc": accuracy_score(y, pred),
                  "macroF1": f1_score(y, pred, average="macro"),
                  "F1_eating": f1_score(y == EAT, pred == EAT), "F1_walking": f1_score(y == WALK, pred == WALK)}
d = res["RankMix"]
print("%-42s acc=%.4f  macroF1=%.4f  <- sụp đổ vì hạng đa lớp không so được" % (d["tag"], d["acc"], d["macroF1"]))

In [ ]:
# ================= BẢNG SO SÁNH TỔNG HỢP + BIỂU ĐỒ =================
order = ["RF_baiBao", "RF_RFE60", "HGB", "OvR", "RankMix"]
tab = pd.DataFrame([{"Cấu hình": res[k]["tag"],
                     "Acc": round(res[k]["acc"], 4), "macro-F1": round(res[k]["macroF1"], 4),
                     "F1 ăn": round(res[k]["F1_eating"], 4), "F1 đi lại": round(res[k]["F1_walking"], 4)} for k in order])
print(tab.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
keys = [k for k in order if k != "RankMix"]
names = [res[k]["tag"].split(" (")[0][:22] for k in keys]
x = np.arange(len(keys)); w = 0.38
axes[0].bar(x - w/2, [res[k]["acc"] for k in keys], w, label="Acc", color="#2e7d32")
axes[0].bar(x + w/2, [res[k]["macroF1"] for k in keys], w, label="macro-F1", color="#fb8c00")
axes[0].set_xticks(x, names, rotation=12); axes[0].set_ylim(0.995, 1.0001)
axes[0].set_title("Acc / macro-F1: chưa sửa vs đã sửa"); axes[0].legend()
axes[1].bar(x - w/2, [res[k]["F1_eating"] for k in keys], w, label="F1 ăn", color="#43a047")
axes[1].bar(x + w/2, [res[k]["F1_walking"] for k in keys], w, label="F1 đi lại", color="#1e88e5")
axes[1].set_xticks(x, names, rotation=12); axes[1].set_ylim(0.98, 1.0005)
axes[1].set_title("Hai lớp hiếm: ăn & đi lại"); axes[1].legend()
plt.tight_layout(); plt.show()

print("\nBáo cáo phân lớp chi tiết của bản được chọn (OvR):")
print(classification_report(y, P_ovr.argmax(1), target_names=list(le.classes_), digits=4))

## ĐỌC KẾT QUẢ
- **Chưa sửa (phương pháp bài báo):** RF cho kết quả tốt trên dữ liệu gộp, nhưng con số
  *công bố per-cow* của bài báo chỉ đạt 83,7–96,5% acc với **nhạy đi lại thường <60%**
  (mỗi bò chỉ có ~1 nghìn mẫu → lớp đi lại 1,8% còn chừng 18 mẫu).
- **Đã sửa:** gộp đàn + HistGradientBoosting đưa F1 đi lại 98,99 → **99,33%**;
  thêm 6 chuyên gia OvR đưa macro-F1 lên mức cao nhất **0,9983**.
- **RankMix sụp (81,5%)** — minh chứng rằng không phải cách ghép nào cũng đúng:
  xếp hạng chỉ hợp lệ khi ghép cùng một bài toán nhị phân.
- Kiểm định thời gian xuôi (train ngày trước, chấm ngày sau) vẫn chỉ ~26% với mọi cấu hình trên:
  lệch thời gian cần **đặc trưng nhịp ngày đêm** (đã phát triển ở vòng v3 của luận văn,
  F1 ăn ngoài phân bố 48,3% → 95,6%). Các sửa ở đây tấn công **lớp hiếm**,
  còn v3 tấn công **lệch thời gian** — hai lớp sửa bổ sung cho nhau.
- Số liệu chi tiết: `results/compare_feeding.json` (Mục 4.8 của luận văn).